<a href="https://colab.research.google.com/github/WilsonForeroB/ColabIA/blob/main/AgenteWebRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔎➡️📚 Agente de búsqueda web + RAG sobre las páginas encontradas

**Objetivo:** construir paso a paso, con **LangChain** y un LLM local (**Ollama**), un sistema en dos fases:

1. **Fase Agente** – Un agente con la herramienta `buscar_en_internet` decide qué buscar, lee los resultados y **devuelve una respuesta** citando las URLs.
2. **Fase RAG** – Las páginas web que encontró el agente se **descargan, se dividen en chunks, se vectorizan y se guardan en FAISS**. Después podemos hacer preguntas de seguimiento que se responden **solo con el contenido real de esas webs**.

```
                 ┌──────────────── FASE 1: AGENTE ────────────────┐
Usuario ──► LLM (qwen3:4b) ──tool_call──► buscar_en_internet (DuckDuckGo)
                 ▲                                │
                 └────────── ToolMessage ◄────────┘   → respuesta + URLs
                 └────────────────────────────────────────────────┘
                                   │ URLs encontradas
                 ┌──────────────── FASE 2: RAG ───────────────────┐
  WebBaseLoader ─► TextSplitter ─► OllamaEmbeddings ─► FAISS
                                                          │ retriever (top-k)
  Pregunta de seguimiento ─► Prompt + contexto ─► LLM ─► Respuesta con fuentes
                 └────────────────────────────────────────────────┘
```

**Stack:** Ollama (`qwen3:4b` + `nomic-embed-text`) · LangChain (`bind_tools`, `WebBaseLoader`, `RecursiveCharacterTextSplitter`, `FAISS`, LCEL) · DuckDuckGo (`ddgs`) · Gradio (opcional).

⚠️ Usa un entorno con GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

## 📋 CELDA 1 — Instalación

Instala Ollama y las librerías de Python. Tarda 2–3 minutos y solo hay que ejecutarla **una vez por sesión**.

In [ ]:
# ==============================================================================
# CELDA 1: INSTALACIÓN
# ==============================================================================
import os
os.environ["DEBIAN_FRONTEND"] = "noninteractive"

print("📦 Dependencias del sistema (zstd para el binario, pciutils para detectar la GPU)...")
!sudo apt-get update -qq 2>/dev/null
!sudo apt-get install -y -qq zstd pciutils 2>/dev/null

print("📦 Instalando Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | grep -v systemd

print("📦 Instalando librerías de Python...")
!pip install -q -U ddgs langchain-core langchain-community langchain-ollama \
                langchain-text-splitters faiss-cpu beautifulsoup4 gradio

print("\n✅ Instalación completada.")

## 📋 CELDA 2 — Arrancar Ollama y descargar los modelos

Usamos **dos modelos**:

| Modelo | Para qué |
|---|---|
| `qwen3:4b` | LLM de chat. Soporta **tool calling** (necesario para el agente) y es bueno en español. |
| `nomic-embed-text` | Modelo de **embeddings**: convierte cada chunk de texto en un vector para la búsqueda semántica del RAG. |

In [ ]:
# ==============================================================================
# CELDA 2: SERVIDOR OLLAMA + MODELOS
# ==============================================================================
import subprocess, time, requests

LLM_MODEL   = "qwen3:4b"          # alternativas con tool calling: llama3.1:8b, qwen3:8b
EMBED_MODEL = "nomic-embed-text"

print("🚀 Iniciando servidor Ollama...")
ollama_proc = subprocess.Popen(["ollama", "serve"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434", timeout=2)
        print("✅ Servidor activo")
        break
    except requests.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("❌ Ollama no arrancó")

print(f"⬇️  Descargando {LLM_MODEL} y {EMBED_MODEL}...")
!ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}
!ollama list

## 📋 CELDA 3 — Configuración

Panel de control del sistema. 💡 **Ejercicio:** cambia `max_resultados`, `chunk_size` o `k` y observa cómo varían las respuestas.

In [ ]:
# ==============================================================================
# CELDA 3: CONFIGURACIÓN — MODIFICA AQUÍ
# ==============================================================================
os.environ["USER_AGENT"] = "Mozilla/5.0 (ColabIA-AgenteWebRAG)"   # WebBaseLoader lo pide

config = {
    "temperatura": 0.1,
    "max_resultados": 5,        # resultados por búsqueda web
    "region": "es-es",          # región de DuckDuckGo
    "max_iteraciones": 4,       # límite del bucle del agente
    "chunk_size": 1000,         # tamaño de cada fragmento (caracteres)
    "chunk_overlap": 150,       # solapamiento entre fragmentos
    "k": 4,                     # nº de chunks que recupera el RAG
}
for k_, v in config.items():
    print(f"   {k_}: {v}")

## 📋 CELDA 4 — La herramienta de búsqueda web

Una **tool** de LangChain es una función de Python con un *docstring* que el LLM lee para decidir **cuándo** y **cómo** usarla.

Además de devolver los resultados al agente, la herramienta **guarda las URLs** en `fuentes_encontradas`: serán la materia prima del RAG en la fase 2.

In [ ]:
# ==============================================================================
# CELDA 4: HERRAMIENTA DE BÚSQUEDA
# ==============================================================================
from ddgs import DDGS
from langchain_core.tools import tool

fuentes_encontradas: dict = {}   # url -> título  (se rellena al buscar)

@tool
def buscar_en_internet(consulta: str) -> str:
    """Busca información actual en internet. Úsala siempre que necesites datos
    recientes o que no conozcas. La entrada es una consulta breve con palabras clave."""
    try:
        resultados = DDGS().text(consulta, region=config["region"],
                                 max_results=config["max_resultados"])
    except Exception as e:
        return f"Error en la búsqueda: {e}"

    if not resultados:
        return "No se encontraron resultados."

    bloques = []
    for i, r in enumerate(resultados, 1):
        fuentes_encontradas[r["href"]] = r["title"]
        bloques.append(f"[{i}] {r['title']}\nURL: {r['href']}\n{r['body']}")
    return "\n\n".join(bloques)

# Prueba directa de la herramienta (sin LLM)
print(buscar_en_internet.invoke("qué es LangChain")[:800])

## 📋 CELDA 5 — FASE 1: el agente (Razonar → Actuar → Observar)

Implementamos el bucle del agente a mano para ver qué ocurre por dentro:

1. `llm.bind_tools([...])` le dice al modelo qué herramientas existen.
2. Si el modelo responde con `tool_calls`, ejecutamos la herramienta y devolvemos el resultado como `ToolMessage`.
3. Repetimos hasta que el modelo responda con texto (respuesta final) o se alcance `max_iteraciones`.

> En LangChain 1.x existe `langchain.agents.create_agent`, que hace este mismo bucle por ti. Aquí lo escribimos explícitamente con fines didácticos.

In [ ]:
# ==============================================================================
# CELDA 5: AGENTE DE BÚSQUEDA WEB
# ==============================================================================
from datetime import date
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# reasoning=False desactiva el modo "thinking" de qwen3 → respuestas más rápidas
llm = ChatOllama(model=LLM_MODEL, temperature=config["temperatura"], reasoning=False)

tools = [buscar_en_internet]
tools_map = {t.name: t for t in tools}
llm_con_tools = llm.bind_tools(tools)

SYSTEM_AGENTE = (
    f"Eres un asistente de investigación. Hoy es {date.today().isoformat()}. "
    "Para responder, usa SIEMPRE la herramienta buscar_en_internet. "
    "Responde en español, de forma concisa, y cita las URLs que utilizaste."
)

def agente_web(pregunta: str, verbose: bool = True) -> str:
    mensajes = [SystemMessage(content=SYSTEM_AGENTE), HumanMessage(content=pregunta)]

    for paso in range(config["max_iteraciones"]):
        ai_msg = llm_con_tools.invoke(mensajes)
        mensajes.append(ai_msg)

        if not ai_msg.tool_calls:                 # → respuesta final
            return ai_msg.content

        for call in ai_msg.tool_calls:            # → ejecutar herramientas
            if verbose:
                print(f"🛠️  Paso {paso+1}: {call['name']}({call['args']})")
            resultado = tools_map[call["name"]].invoke(call["args"])
            mensajes.append(ToolMessage(content=resultado, tool_call_id=call["id"]))

    # Si se agotan las iteraciones, pedimos una respuesta con lo que tenga
    mensajes.append(HumanMessage(content="Responde ya con la información obtenida."))
    return llm.invoke(mensajes).content

print("✅ Agente listo.")

### ▶️ Probar el agente

Cambia `PREGUNTA` por un tema de tu interés. Fíjate en las URLs que se acumulan en `fuentes_encontradas`.

In [ ]:
PREGUNTA = "¿Qué novedades tiene la última versión de LangChain?"   # 👈 cámbiala

fuentes_encontradas.clear()
respuesta = agente_web(PREGUNTA)

print("\n" + "=" * 70)
print("🤖 RESPUESTA DEL AGENTE\n")
print(respuesta)
print("=" * 70)
print(f"\n🌐 {len(fuentes_encontradas)} páginas encontradas:")
for url, titulo in fuentes_encontradas.items():
    print(f"   • {titulo}\n     {url}")

## 📋 CELDA 6 — FASE 2a: construir el RAG con esas webs

Los resultados del buscador solo traen un *snippet* de cada página. Ahora descargamos **la página completa**:

| Paso | Componente LangChain | Qué hace |
|---|---|---|
| Cargar | `WebBaseLoader` | Descarga el HTML y extrae el texto (BeautifulSoup) |
| Dividir | `RecursiveCharacterTextSplitter` | Corta el texto en chunks con solapamiento |
| Vectorizar | `OllamaEmbeddings` | Convierte cada chunk en un vector |
| Indexar | `FAISS` | Base vectorial para buscar por similitud semántica |

In [ ]:
# ==============================================================================
# CELDA 6: CONSTRUIR EL ÍNDICE VECTORIAL A PARTIR DE LAS URLs
# ==============================================================================
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

def construir_rag(urls: list):
    # 1) Cargar: una URL cada vez, para que una web caída o que bloquee
    #    a los bots no rompa todo el proceso
    docs = []
    for url in urls:
        try:
            loader = WebBaseLoader(url, raise_for_status=True,
                                   requests_kwargs={"timeout": 15})
            docs += [d for d in loader.load() if len(d.page_content.strip()) > 200]
        except Exception as e:
            print(f"   ⚠️  No se pudo cargar {url}: {type(e).__name__}")
    for d in docs:   # limpiar espacios y líneas vacías repetidas
        d.page_content = "\n".join(l.strip() for l in d.page_content.splitlines() if l.strip())
    print(f"📄 Páginas descargadas con contenido: {len(docs)}/{len(urls)}")
    if not docs:
        raise ValueError("Ninguna página pudo descargarse. Prueba con otra pregunta.")

    # 2) Dividir
    splitter = RecursiveCharacterTextSplitter(chunk_size=config["chunk_size"],
                                              chunk_overlap=config["chunk_overlap"])
    chunks = splitter.split_documents(docs)
    print(f"✂️  Chunks generados: {len(chunks)}")

    # 3) + 4) Vectorizar e indexar
    vs = FAISS.from_documents(chunks, embeddings)
    print(f"🗄️  Índice FAISS creado con {vs.index.ntotal} vectores")
    return vs

vectorstore = construir_rag(list(fuentes_encontradas))
retriever = vectorstore.as_retriever(search_kwargs={"k": config["k"]})

### 🔍 ¿Qué recupera el retriever?

Antes de pasarle nada al LLM, miramos qué chunks considera más relevantes para una pregunta. Así se entiende la "R" de RAG.

In [ ]:
for i, d in enumerate(retriever.invoke("principales novedades"), 1):   # 👈 prueba otras consultas
    print(f"--- Chunk {i} · {d.metadata.get('source')}")
    print(d.page_content[:300], "...\n")

## 📋 CELDA 7 — FASE 2b: cadena RAG con LCEL

Encadenamos con el operador `|`: **retriever → formateo del contexto → prompt → LLM → texto**.

El prompt obliga al modelo a responder **solo** con el contexto recuperado y a decir "no lo sé" si la información no está, lo que reduce las alucinaciones.

In [ ]:
# ==============================================================================
# CELDA 7: CADENA RAG
# ==============================================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

prompt_rag = ChatPromptTemplate.from_messages([
    ("system",
     "Responde a la pregunta usando ÚNICAMENTE el contexto, extraído de páginas web. "
     "Si la respuesta no está en el contexto, di que no aparece en las fuentes. "
     "Responde en español y cita entre corchetes la URL de la que sale cada dato.\n\n"
     "CONTEXTO:\n{contexto}"),
    ("human", "{pregunta}"),
])

def formatear_docs(docs) -> str:
    return "\n\n".join(f"[{d.metadata.get('source')}]\n{d.page_content}" for d in docs)

def crear_cadena(retriever):
    return (
        {"contexto": retriever | RunnableLambda(formatear_docs),
         "pregunta": RunnablePassthrough()}
        | prompt_rag
        | llm
        | StrOutputParser()
    )

cadena_rag = crear_cadena(retriever)

def preguntar_rag(pregunta: str) -> str:
    respuesta = cadena_rag.invoke(pregunta)
    fuentes = sorted({d.metadata.get("source") for d in retriever.invoke(pregunta)})
    return respuesta + "\n\n📚 Fuentes consultadas:\n" + "\n".join(f"• {u}" for u in fuentes)

print("✅ Cadena RAG lista.")

### ▶️ Preguntas de seguimiento sobre las webs indexadas

Compara con la respuesta del agente: ahora el modelo trabaja con **el texto completo** de las páginas, no con los snippets del buscador.

In [ ]:
print(preguntar_rag("Resume los puntos más importantes que mencionan las fuentes."))   # 👈 cámbiala

In [ ]:
# Pregunta fuera de tema → debería decir que no aparece en las fuentes
print(preguntar_rag("¿Cuál es la receta de la paella valenciana?"))

## 📋 CELDA 8 — (Opcional) Interfaz Gradio con las dos fases

* **Pestaña 1:** el agente busca en internet, responde y construye automáticamente el índice RAG con las webs encontradas.
* **Pestaña 2:** chat de seguimiento sobre ese índice.

Gradio genera un enlace público temporal (`share=True`).

In [ ]:
# ==============================================================================
# CELDA 8: INTERFAZ GRADIO
# ==============================================================================
import gradio as gr

def ui_buscar(pregunta: str):
    global vectorstore, retriever, cadena_rag
    fuentes_encontradas.clear()
    respuesta = agente_web(pregunta)
    if not fuentes_encontradas:
        return respuesta, "⚠️ El agente no buscó en internet; no hay webs que indexar."
    try:
        vectorstore = construir_rag(list(fuentes_encontradas))
        retriever = vectorstore.as_retriever(search_kwargs={"k": config["k"]})
        cadena_rag = crear_cadena(retriever)
        estado = (f"✅ RAG construido con {len(fuentes_encontradas)} webs "
                  f"({vectorstore.index.ntotal} chunks). Ve a la pestaña 2.\n\n"
                  + "\n".join(f"• {u}" for u in fuentes_encontradas))
    except Exception as e:
        estado = f"⚠️ No se pudo construir el RAG: {e}"
    return respuesta, estado

def ui_chat_rag(mensaje, historial):
    return preguntar_rag(mensaje)

with gr.Blocks(title="Agente Web + RAG", theme="soft") as demo:
    gr.Markdown("# 🔎 Agente de búsqueda web + 📚 RAG")
    with gr.Tab("1 · Buscar en internet"):
        entrada = gr.Textbox(label="Pregunta", placeholder="¿Qué es el protocolo MCP?")
        boton = gr.Button("Buscar y construir RAG", variant="primary")
        salida = gr.Markdown(label="Respuesta del agente")
        estado = gr.Textbox(label="Estado del RAG", lines=6)
        boton.click(ui_buscar, inputs=entrada, outputs=[salida, estado])
    with gr.Tab("2 · Preguntar a las webs (RAG)"):
        gr.ChatInterface(fn=ui_chat_rag, type="messages")

demo.launch(share=True, debug=False)

## 📋 CELDA 9 — Cierre y limpieza

Ejecútala al terminar para liberar recursos.

In [ ]:
# ==============================================================================
# CELDA 9: LIMPIEZA
# ==============================================================================
try:
    demo.close(); print("✅ Gradio cerrado")
except Exception as e:
    print(f"⚠️  Gradio: {e}")

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
print("✅ Ollama detenido")
fuentes_encontradas.clear()
print("🏁 Sesión terminada.")

## 🎓 Resumen y retos

| Concepto | Dónde lo vimos |
|---|---|
| Tool calling (`@tool`, `bind_tools`, `ToolMessage`) | Celdas 4–5 |
| Bucle de agente ReAct | Celda 5 |
| Carga de webs (`WebBaseLoader`) | Celda 6 |
| Chunking + embeddings + FAISS | Celda 6 |
| Cadena RAG con LCEL | Celda 7 |

**Retos:**
1. Añade una segunda herramienta `buscar_en_sitio(consulta, dominio)` que use `site:dominio`.
2. Guarda el índice con `vectorstore.save_local("indice")` en Google Drive y recárgalo en otra sesión con `FAISS.load_local`.
3. Convierte el RAG en **otra herramienta del agente** (`consultar_webs_indexadas`) para que decida él solo si buscar en internet o en el índice.
4. Añade memoria conversacional a la pestaña 2 pasando el `historial` al prompt.